# Threat hunting playbook

A minimal **watchlist + join + timeline** hunt on bundled **`cribl_search_sample`** VPC flow telemetry.

1. Pick **six external (non-private) destination IPs** from VPC flows
2. Save them as a **static Search lookup**
3. **Join** flows to the watchlist and aggregate with **`timestats`**
4. **Chart** when those IPs appear on the timeline

## Hunt charter

**Hypothesis:** A small set of high-volume **external destination IPs** in VPC flow logs are worth tracking; joining a static lookup surfaces *when* they show up in the sample dataset.

**Data:** `dataset=cribl_search_sample` with `dataSource == "vpcflowlogs"` (VPC flow records — the sample’s VPC flow slice).

**Artifacts:** `notebook_vpc_external_watchlist.csv` (lookup; removed in **Cleanup**).

## Prerequisites

1. Hosted Cribl app with **Cribl Search** and lookup magics (`%%cribl_save_search_lookup`, etc.).
2. No external URLs or dataset providers — everything uses **`cribl_search_sample`**.

**Related:** `Cribl_Search_Lookup_Magics.ipynb`, `Cribl_Search_Examples.ipynb`, `Incident_Triage_Playbook.ipynb`.

### A) Select six external destination IPs

From VPC flows, rank **`dstaddr`** values that are valid IPv4 and **not** in private ranges (`ipv4_is_private`).

In [ ]:
%%cribl_search var=external_ip_candidates lang=kql limit=50 preview=true earliest=-7d latest=now timeout=180
dataset=cribl_search_sample
| where dataSource == "vpcflowlogs"
| where isnotnull(parse_ipv4(tostring(dstaddr))) and not(ipv4_is_private(tostring(dstaddr)))
| summarize flow_count = count() by ip_address = tostring(dstaddr)
| top 6 by flow_count desc

### B) Build and publish the static lookup

In [ ]:
import pandas as pd

def _pick_col(frame, *names):
    keys = {str(c).strip().lower(): c for c in frame.columns}
    for n in names:
        if n in keys:
            return keys[n]
    return None

ip_col = _pick_col(external_ip_candidates, "ip_address", "dstaddr")
count_col = _pick_col(external_ip_candidates, "flow_count", "count")
if ip_col is None:
    raise ValueError(
        f"Expected ip_address from Search; got {list(external_ip_candidates.columns)}. Re-run §A."
    )
if len(external_ip_candidates) == 0:
    raise ValueError("No external IPs returned — check dataSource filter or Search availability.")

watchlist_df = external_ip_candidates[[ip_col] + ([count_col] if count_col else [])].copy()
watchlist_df = watchlist_df.rename(columns={ip_col: "ip_address"})
if count_col:
    watchlist_df = watchlist_df.rename(columns={count_col: "flow_count"})
watchlist_df["watchlist"] = "vpc_external_dst"
watchlist_df = watchlist_df.drop_duplicates(subset=["ip_address"]).head(6)

print("watchlist rows:", len(watchlist_df))
watchlist_df

In [ ]:
%%cribl_save_search_lookup notebook_vpc_external_watchlist.csv var=watchlist_df replace=true

In [ ]:
%%cribl_search var=lookup_preview lang=kql limit=10 preview=true earliest=-7d latest=now
dataset="$vt_lookups" lookupFile="notebook_vpc_external_watchlist"
| limit 10

### C) Join VPC flows to the watchlist + `timestats`

**Inner join** to `$vt_lookups` on `dstaddr`. Cribl **`timestats`** uses **`span=` before** the aggregation (`timestats span=1h count() by ip_address`). Do not start the query with `let` — the notebook adds a `cribl` prefix to `dataset=` pipelines only.

In [ ]:
%%cribl_search var=timeline lang=kql limit=5000 preview=true earliest=-7d latest=now timeout=300
dataset=cribl_search_sample
| where dataSource == "vpcflowlogs"
| join kind=inner (
    dataset="$vt_lookups" lookupFile="notebook_vpc_external_watchlist"
  ) on $left.dstaddr == $right.ip_address
| timestats span=1h count() by ip_address

### D) Visualize timeline

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def _pick_col(frame, *names):
    keys = {str(c).strip().lower(): c for c in frame.columns}
    for n in names:
        if n in keys:
            return keys[n]
    return None

if len(timeline) == 0:
    raise ValueError("timeline is empty — re-run §A–C (lookup must exist before join).")

time_col = _pick_col(timeline, "_time", "time", "timestamp")
ip_col = _pick_col(timeline, "ip_address", "dstaddr")
count_col = _pick_col(timeline, "count_", "count", "flows")

if time_col is None or ip_col is None:
    raise ValueError(f"Expected _time and ip_address; got {list(timeline.columns)}")

plot_df = timeline[[time_col, ip_col] + ([count_col] if count_col else [])].copy()
plot_df[time_col] = pd.to_datetime(plot_df[time_col], errors="coerce")
plot_df = plot_df.dropna(subset=[time_col])
if count_col:
    plot_df[count_col] = pd.to_numeric(plot_df[count_col], errors="coerce").fillna(0)
else:
    count_col = "_n"
    plot_df[count_col] = 1

print("timestats buckets:", len(plot_df), "| IPs:", plot_df[ip_col].nunique())

fig, ax = plt.subplots(figsize=(10, 4))
for ip, grp in plot_df.groupby(ip_col):
    g = grp.sort_values(time_col)
    ax.plot(g[time_col], g[count_col], marker="o", linewidth=1.5, label=str(ip))
ax.set_title("Watchlisted external IPs — flow hits over time (timestats 1h)")
ax.set_xlabel("Time")
ax.set_ylabel("Flow count")
ax.legend(loc="upper right", fontsize=8, ncol=2)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(
    "Verdict: joined VPC flows show when each watchlisted external IP appears in the sample window."
)
plot_df.head(12)

### Cleanup

In [ ]:
%%cribl_delete_search_lookup notebook_vpc_external_watchlist.csv

### Next steps

- Swap `dataSource == "vpcflowlogs"` for your production VPC dataset name
- Add `srcaddr` join (union) if you also want hits when watchlisted IPs are sources
- `Cribl_Search_Lookup_Magics.ipynb` — lookup lifecycle
- `Cribl_Search_Examples.ipynb` — more KQL patterns